# 🎯 High-Yield Balanced 6,400 Dataset Generation & Labeling
### Targeting the Big 3 Scholars (Ibn Baz, Ibn Uthaymeen, Al-Lajnah) to Reach 3,200+ Positives

This notebook scales your dataset from **4.3k $ightarrow$ 6,400+ perfectly balanced examples (50% positive / 50% negative)**:
1. **The Strategy**:
   - Instead of asking minor scholars questions they never ruled on, we target the **Big 3 Prolific Scholars** (Ibn Baz, Ibn Uthaymeen, Al-Lajnah Ad-Da'imah) who hold 92% of the corpus.
   - We query them with **200 targeted jurisprudential questions** (embedded in Cell 3).
2. **Quota (8 Candidates / Question)**:
   - Sheikh Ibn Baz: 4 candidates
   - Sheikh Ibn Uthaymeen: 2 candidates
   - Al-Lajnah Ad-Da'imah: 2 candidates
   - *Total: 200 × 8 = 1,600 new candidates.*
3. **High Positive Yield**:
   - The Big 3 have a ~70% positive hit rate in these domains $ightarrow$ **~1,100 to 1,200 new positive fatwas**!
4. **Targeted Labeling**:
   - Ingests your previous labeled dataset (which already has 2,049 positives).
   - Labels **ONLY the 1,600 new candidates** with GPT-4o Mini (Cost: **~$0.15**, Time: ~2.5 mins).
5. **Auto-Balancing to 6,400+ Examples**:
   - Combines 2,049 (old) + ~1,150 (new) = **~3,200 total positives**.
   - Pairs them with the **3,200 hardest negatives** from Mizan reranking.
   - Produces `dataset_6k_balanced_bundle.zip` (50% positive / 50% negative)!


In [ ]:
# 1. Install & Verify Dependencies
!pip install -q sentence-transformers python-docx

import os
import sys
import time
import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

print("PyTorch Version:         ", torch.__version__)
print("CUDA Available:          ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:              ", torch.cuda.get_device_name(0))


In [ ]:
# 2. Download Multi-Scholar Corpus & Precomputed Embeddings from Hugging Face
from huggingface_hub import hf_hub_download

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not token:
    token = os.getenv("HF_TOKEN")

DATA_REPO_ID = "ZenoAbderrahim/binbaz_fatwas_rag"
DATA_DIR = Path("/kaggle/working/data") if Path("/kaggle/working").exists() else Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILES_TO_DOWNLOAD = [
    "multi_scholar_clean.csv",
    "qwen3_06b_multi_scholar_embeddings.pt",
]

print(f"Syncing corpus files from Hugging Face dataset: '{DATA_REPO_ID}'...")
for fname in FILES_TO_DOWNLOAD:
    dest = DATA_DIR / fname
    if not dest.exists():
        print(f"  • Downloading {fname} ...")
        hf_hub_download(repo_id=DATA_REPO_ID, repo_type="dataset", filename=fname, local_dir=DATA_DIR, token=token)
    else:
        print(f"  ✓ {fname} already present.")

print("\nLoading cleaned fatwa corpus...")
corpus_df = pd.read_csv(DATA_DIR / "multi_scholar_clean.csv")
print(f"✓ Total corpus size: {len(corpus_df):,} fatwas across {corpus_df['source'].nunique()} scholars.")

print("\nLoading precomputed Qwen 0.6B corpus embeddings...")
corpus_embeddings = torch.load(DATA_DIR / "qwen3_06b_multi_scholar_embeddings.pt", map_location="cpu", weights_only=False)
print(f"✓ Corpus embeddings shape: {corpus_embeddings.shape}")


In [ ]:
# 3. Load the 200 Questions Targeting the Big 3 Scholars
# (Embedded directly in code to guarantee 100% fidelity, IDs 751..950)
QUESTIONS_200 = [
  {
    "question_id": 751,
    "question": "ما حكم الوضوء إذا لمس الرجل فرجه مباشرة دون حائل؟"
  },
  {
    "question_id": 752,
    "question": "هل ينتقض الوضوء بلمس المرأة الأجنبية؟"
  },
  {
    "question_id": 753,
    "question": "ما حكم الوضوء إذا خرج الدم من الأنف أو اللثة؟"
  },
  {
    "question_id": 754,
    "question": "هل خروج المذي يوجب الغسل أم الوضوء فقط؟"
  },
  {
    "question_id": 755,
    "question": "ما حكم من شك في انتقاض وضوئه بعد الفراغ من الصلاة؟"
  },
  {
    "question_id": 756,
    "question": "هل يجوز المسح على الجوربين إذا كانا رقيقين؟"
  },
  {
    "question_id": 757,
    "question": "ما مدة المسح على الخفين للمقيم والمسافر؟"
  },
  {
    "question_id": 758,
    "question": "هل يجوز المسح على الجوربين إذا لبسهما المسلم بعد وضوء غير مكتمل؟"
  },
  {
    "question_id": 759,
    "question": "ما حكم من نسي غسل عضو من أعضاء الوضوء ثم تذكر بعد الصلاة؟"
  },
  {
    "question_id": 760,
    "question": "هل يصح الوضوء مع وجود طلاء يمنع وصول الماء إلى الأظافر؟"
  },
  {
    "question_id": 761,
    "question": "ما حكم استعمال الماء المتغير بالصابون في الوضوء؟"
  },
  {
    "question_id": 762,
    "question": "هل يجب إزالة النجاسة عن الثوب قبل الوضوء أم قبل الصلاة؟"
  },
  {
    "question_id": 763,
    "question": "ما حكم من صلى ثم اكتشف وجود نجاسة على ثوبه؟"
  },
  {
    "question_id": 764,
    "question": "هل يكفي غسل موضع النجاسة مرة واحدة لإزالتها؟"
  },
  {
    "question_id": 765,
    "question": "ما حكم الوضوء من ماء خالطه شيء طاهر فغير لونه؟"
  },
  {
    "question_id": 766,
    "question": "هل يجوز التيمم مع وجود الماء إذا كان استعماله يضر بالمريض؟"
  },
  {
    "question_id": 767,
    "question": "كيف يتيمم المسلم إذا لم يجد تراباً؟"
  },
  {
    "question_id": 768,
    "question": "ما حكم من صلى بالتيمم ثم وجد الماء قبل خروج وقت الصلاة؟"
  },
  {
    "question_id": 769,
    "question": "هل يجب الغسل على من احتلم ولم يجد أثراً للمني؟"
  },
  {
    "question_id": 770,
    "question": "ما حكم من اغتسل من الجنابة ثم شك هل عمم الماء جميع جسده؟"
  },
  {
    "question_id": 771,
    "question": "هل يجوز تأخير غسل الجنابة إلى ما بعد طلوع الفجر في رمضان؟"
  },
  {
    "question_id": 772,
    "question": "ما حكم الصلاة لمن كان على جنابة ونسي الاغتسال؟"
  },
  {
    "question_id": 773,
    "question": "هل يجب على المرأة الغسل إذا رأت في المنام ما يراه الرجل؟"
  },
  {
    "question_id": 774,
    "question": "ما حكم الإفرازات التي تخرج من المرأة وهل تنقض الوضوء؟"
  },
  {
    "question_id": 775,
    "question": "هل يجوز للمحدث قراءة القرآن من الهاتف دون لمس المصحف؟"
  },
  {
    "question_id": 776,
    "question": "ما حكم الصلاة بثوب أصابته قطرات من البول؟"
  },
  {
    "question_id": 777,
    "question": "هل يصح الوضوء إذا نسي المسلم التسمية في أوله؟"
  },
  {
    "question_id": 778,
    "question": "ما حكم من توضأ ثم نام جالساً ولم يستند إلى شيء؟"
  },
  {
    "question_id": 779,
    "question": "هل ينتقض الوضوء بأكل لحم الإبل؟"
  },
  {
    "question_id": 780,
    "question": "ما حكم من توضأ ثم قص أظافره أو شعره؟"
  },
  {
    "question_id": 781,
    "question": "ما حكم من ترك الصلاة تكاسلاً مع إقراره بوجوبها؟"
  },
  {
    "question_id": 782,
    "question": "هل يجب قضاء الصلاة التي فاتت بسبب النوم؟"
  },
  {
    "question_id": 783,
    "question": "ما حكم من استيقظ بعد طلوع الشمس ولم يصل الفجر؟"
  },
  {
    "question_id": 784,
    "question": "هل تصح الصلاة إذا صلى المسلم وفي جيبه هاتف يصدر صوتاً؟"
  },
  {
    "question_id": 785,
    "question": "ما حكم الصلاة في مكان العمل إذا لم يوجد مصلى مخصص؟"
  },
  {
    "question_id": 786,
    "question": "هل يجوز تأخير الصلاة إلى آخر وقتها بسبب الانشغال بالدراسة؟"
  },
  {
    "question_id": 787,
    "question": "ما حكم من صلى ثم تذكر أنه لم يكن متوضئاً؟"
  },
  {
    "question_id": 788,
    "question": "هل تصح الصلاة إذا انكشف جزء من عورة المصلي دون أن ينتبه؟"
  },
  {
    "question_id": 789,
    "question": "ما حكم الصلاة بثياب عليها صور ذوات الأرواح؟"
  },
  {
    "question_id": 790,
    "question": "هل يجوز الصلاة في غرفة فيها صور معلقة على الجدران؟"
  },
  {
    "question_id": 791,
    "question": "ما حكم من صلى وهو يدافع البول أو الغائط؟"
  },
  {
    "question_id": 792,
    "question": "هل يجوز للمريض أن يصلي جالساً إذا كان القيام يشق عليه؟"
  },
  {
    "question_id": 793,
    "question": "كيف يصلي من لا يستطيع الركوع أو السجود؟"
  },
  {
    "question_id": 794,
    "question": "ما حكم من عجز عن استقبال القبلة بسبب المرض؟"
  },
  {
    "question_id": 795,
    "question": "هل تصح صلاة من أخطأ في قراءة الفاتحة دون أن يغير المعنى؟"
  },
  {
    "question_id": 796,
    "question": "ما حكم قراءة الفاتحة للمأموم خلف الإمام في الصلاة الجهرية؟"
  },
  {
    "question_id": 797,
    "question": "هل يجوز للمسبوق أن يقوم لإتمام صلاته قبل تسليم الإمام التسليمة الثانية؟"
  },
  {
    "question_id": 798,
    "question": "ما حكم من أدرك الإمام راكعاً فهل تحسب له الركعة؟"
  },
  {
    "question_id": 799,
    "question": "هل تصح صلاة الجماعة إذا وقف المأموم وحده خلف الصف؟"
  },
  {
    "question_id": 800,
    "question": "ما حكم من سبق الإمام في الركوع أو السجود؟"
  },
  {
    "question_id": 801,
    "question": "هل يجوز للمأموم أن يرفع رأسه من الركوع قبل الإمام سهواً؟"
  },
  {
    "question_id": 802,
    "question": "ما حكم من نسي التشهد الأول وقام إلى الركعة الثالثة؟"
  },
  {
    "question_id": 803,
    "question": "متى يجب سجود السهو قبل السلام ومتى يكون بعده؟"
  },
  {
    "question_id": 804,
    "question": "نسيت سجود السهو بعد الصلاة، فماذا أفعل؟"
  },
  {
    "question_id": 805,
    "question": "ما حكم من شك هل صلى ثلاث ركعات أم أربعاً؟"
  },
  {
    "question_id": 806,
    "question": "إذا شككت في عدد الركعات وغلب على ظني عدد معين، فماذا أفعل؟"
  },
  {
    "question_id": 807,
    "question": "ما حكم من سلم من الصلاة قبل إتمامها ثم تذكر بعد دقائق؟"
  },
  {
    "question_id": 808,
    "question": "هل تبطل الصلاة بكثرة الحركة بسبب الحاجة؟"
  },
  {
    "question_id": 809,
    "question": "ما حكم الالتفات في الصلاة بسبب سماع صوت مفاجئ؟"
  },
  {
    "question_id": 810,
    "question": "هل يجوز للمصلي أن يغمض عينيه أثناء الصلاة إذا كان ذلك يساعده على الخشوع؟"
  },
  {
    "question_id": 811,
    "question": "ما حكم رفع اليدين عند الدعاء بعد الصلاة المفروضة؟"
  },
  {
    "question_id": 812,
    "question": "هل يشرع الجهر بالذكر بعد الصلاة أم الإسرار به؟"
  },
  {
    "question_id": 813,
    "question": "ما حكم من فاتته صلاة الوتر ثم تذكرها في النهار؟"
  },
  {
    "question_id": 814,
    "question": "هل يجوز صلاة النافلة بعد صلاة العصر؟"
  },
  {
    "question_id": 815,
    "question": "ما حكم صلاة ركعتين عند دخول المسجد أثناء خطبة الجمعة؟"
  },
  {
    "question_id": 816,
    "question": "هل يجوز الكلام أثناء خطبة الجمعة إذا كان الكلام لنصح شخص؟"
  },
  {
    "question_id": 817,
    "question": "ما حكم من تأخر عن الجمعة حتى أدرك الإمام في التشهد الأخير؟"
  },
  {
    "question_id": 818,
    "question": "هل تجب صلاة الجمعة على المسافر؟"
  },
  {
    "question_id": 819,
    "question": "ما حكم من صلى الجمعة ثم اكتشف أنه صلى في غير اتجاه القبلة؟"
  },
  {
    "question_id": 820,
    "question": "هل يجوز جمع الصلاة بسبب المطر أو شدة البرد؟"
  },
  {
    "question_id": 821,
    "question": "هل يجوز للصائم تذوق الطعام أثناء الطبخ دون ابتلاعه؟"
  },
  {
    "question_id": 822,
    "question": "ما حكم من أكل ناسياً في نهار رمضان؟"
  },
  {
    "question_id": 823,
    "question": "إذا شربت الماء ناسياً وأنا صائم، فهل يجب علي قضاء ذلك اليوم؟"
  },
  {
    "question_id": 824,
    "question": "ما حكم من أكل بعد الفجر ظاناً أن وقت السحور لم ينته؟"
  },
  {
    "question_id": 825,
    "question": "ما حكم من أفطر ظاناً أن الشمس غربت ثم تبين أنها لم تغرب؟"
  },
  {
    "question_id": 826,
    "question": "هل يبطل الصوم باستعمال بخاخ الربو في نهار رمضان؟"
  },
  {
    "question_id": 827,
    "question": "ما حكم استخدام قطرة العين أثناء الصيام؟"
  },
  {
    "question_id": 828,
    "question": "هل يفطر الصائم إذا أخذ إبرة علاجية في النهار؟"
  },
  {
    "question_id": 829,
    "question": "ما حكم سحب الدم للتحليل أثناء الصيام؟"
  },
  {
    "question_id": 830,
    "question": "هل يفطر الصائم إذا احتلم أثناء النهار؟"
  },
  {
    "question_id": 831,
    "question": "ما حكم من نام طوال نهار رمضان ولم يستيقظ إلا بعد المغرب؟"
  },
  {
    "question_id": 832,
    "question": "هل يجوز للصائم استعمال السواك بعد الظهر؟"
  },
  {
    "question_id": 833,
    "question": "ما حكم بلع الريق أثناء الصيام؟"
  },
  {
    "question_id": 834,
    "question": "هل يبطل الصوم إذا دخل الماء إلى الحلق أثناء المضمضة بغير قصد؟"
  },
  {
    "question_id": 835,
    "question": "ما حكم من تعمد القيء في نهار رمضان؟"
  },
  {
    "question_id": 836,
    "question": "هل يجب على من أفطر بسبب المرض قضاء الأيام بعد شفائه؟"
  },
  {
    "question_id": 837,
    "question": "ما حكم المريض الذي لا يستطيع الصيام ولا يرجى شفاؤه؟"
  },
  {
    "question_id": 838,
    "question": "هل يجوز للحامل أن تفطر في رمضان خوفاً على جنينها؟"
  },
  {
    "question_id": 839,
    "question": "ما حكم المرأة التي طهرت من الحيض قبل الفجر ولم تغتسل إلا بعده؟"
  },
  {
    "question_id": 840,
    "question": "هل يصح صوم المرأة إذا نزل عليها الحيض قبل غروب الشمس بدقائق؟"
  },
  {
    "question_id": 841,
    "question": "هل تلزم الكفارة لمن جامع زوجته في رمضان وهو مسافر؟"
  },
  {
    "question_id": 842,
    "question": "ما حكم من أفطر عمداً في رمضان ثم تاب إلى الله؟"
  },
  {
    "question_id": 843,
    "question": "هل يجوز صيام التطوع دون تبييت النية من الليل؟"
  },
  {
    "question_id": 844,
    "question": "ما حكم صيام ستة أيام من شوال قبل قضاء رمضان؟"
  },
  {
    "question_id": 845,
    "question": "هل يجوز تأخير قضاء رمضان إلى ما قبل رمضان التالي بأيام قليلة؟"
  },
  {
    "question_id": 846,
    "question": "ما حكم من نذر صيام أيام محددة ثم عجز عن صيامها بسبب المرض؟"
  },
  {
    "question_id": 847,
    "question": "هل يجوز للمسافر أن يفطر إذا بدأ يومه صائماً ثم سافر؟"
  },
  {
    "question_id": 848,
    "question": "ما حكم صيام يوم الجمعة منفرداً؟"
  },
  {
    "question_id": 849,
    "question": "هل يجوز صيام يوم الشك قبل رمضان؟"
  },
  {
    "question_id": 850,
    "question": "ما حكم زكاة المال المدخر لشراء منزل للسكن؟"
  },
  {
    "question_id": 851,
    "question": "هل تجب الزكاة في الذهب الذي تلبسه المرأة للزينة؟"
  },
  {
    "question_id": 852,
    "question": "ما حكم تأخير إخراج الزكاة بعد وجوبها دون عذر؟"
  },
  {
    "question_id": 853,
    "question": "هل يجوز إعطاء الزكاة للأخ الفقير؟"
  },
  {
    "question_id": 854,
    "question": "ما حكم دفع الزكاة إلى جمعية خيرية لتوزعها على المستحقين؟"
  },
  {
    "question_id": 855,
    "question": "هل يجوز إخراج زكاة الفطر نقداً بدلاً من الطعام؟"
  },
  {
    "question_id": 856,
    "question": "متى يجب إخراج زكاة الفطر؟"
  },
  {
    "question_id": 857,
    "question": "هل يجوز دفع الزكاة إلى شخص عليه دين لا يستطيع سداده؟"
  },
  {
    "question_id": 858,
    "question": "ما حكم إعطاء الزكاة للأب أو الأم إذا كانا فقيرين؟"
  },
  {
    "question_id": 859,
    "question": "هل يجوز التصدق من مال الزوجة دون إذنها؟"
  },
  {
    "question_id": 860,
    "question": "ما حكم شراء سلعة بالتقسيط بسعر أعلى من سعرها نقداً؟"
  },
  {
    "question_id": 861,
    "question": "هل يجوز شراء سيارة عن طريق قرض بنكي بفائدة؟"
  },
  {
    "question_id": 862,
    "question": "ما حكم العمل في بنك يتعامل بالربا؟"
  },
  {
    "question_id": 863,
    "question": "هل يجوز إيداع المال في بنك تقليدي عند عدم وجود بنك إسلامي؟"
  },
  {
    "question_id": 864,
    "question": "ما حكم أخذ فوائد الحساب البنكي ثم التخلص منها؟"
  },
  {
    "question_id": 865,
    "question": "هل يجوز البيع بالتقسيط مع اشتراط غرامة عند التأخر في السداد؟"
  },
  {
    "question_id": 866,
    "question": "ما حكم شراء الذهب بالذهب مع دفع فرق السعر؟"
  },
  {
    "question_id": 867,
    "question": "هل يجوز شراء الذهب ببطاقة مصرفية مع تأخر قيد المبلغ؟"
  },
  {
    "question_id": 868,
    "question": "ما حكم بيع سلعة قبل أن يملكها البائع؟"
  },
  {
    "question_id": 869,
    "question": "هل يجوز بيع شيء مع إخفاء عيب يعرفه البائع؟"
  },
  {
    "question_id": 870,
    "question": "ما حق المشتري إذا اكتشف عيباً في السلعة بعد شرائها؟"
  },
  {
    "question_id": 871,
    "question": "هل يجوز أخذ عمولة مقابل الوساطة في بيع العقارات؟"
  },
  {
    "question_id": 872,
    "question": "ما حكم العربون إذا تراجع المشتري عن إتمام البيع؟"
  },
  {
    "question_id": 873,
    "question": "هل يجوز للموظف قبول هدية من عميل له معاملة عنده؟"
  },
  {
    "question_id": 874,
    "question": "ما حكم العمل في شركة تتعامل في منتجات مباحة وأخرى محرمة؟"
  },
  {
    "question_id": 875,
    "question": "هل يجوز أخذ قرض من شخص بشرط أن يقدم له خدمة مجانية؟"
  },
  {
    "question_id": 876,
    "question": "ما حكم شراء أسهم شركة لا يعرف المشتري طبيعة جميع أنشطتها؟"
  },
  {
    "question_id": 877,
    "question": "هل يجوز الاستثمار في شركة تقترض بالربا؟"
  },
  {
    "question_id": 878,
    "question": "ما حكم الاشتراك في مسابقة يدفع فيها المشاركون رسوماً للفوز بجائزة؟"
  },
  {
    "question_id": 879,
    "question": "هل يجوز العمل بالتسويق بالعمولة إذا كان المنتج مباحاً؟"
  },
  {
    "question_id": 880,
    "question": "ما حكم عقد الزواج دون موافقة ولي المرأة؟"
  },
  {
    "question_id": 881,
    "question": "هل يصح الزواج إذا لم يشهد العقد شاهدان؟"
  },
  {
    "question_id": 882,
    "question": "ما حكم تأخير دفع المهر المتفق عليه؟"
  },
  {
    "question_id": 883,
    "question": "هل يجوز للزوجة أن تتنازل عن مهرها بعد عقد الزواج؟"
  },
  {
    "question_id": 884,
    "question": "ما حكم الزواج من امرأة لا تصلي؟"
  },
  {
    "question_id": 885,
    "question": "هل يجوز للمرأة أن تشترط في عقد الزواج ألا ينقلها زوجها من بلدها؟"
  },
  {
    "question_id": 886,
    "question": "ما حكم إجبار المرأة على الزواج ممن لا تريده؟"
  },
  {
    "question_id": 887,
    "question": "هل يجوز للخاطب أن يرى مخطوبته قبل عقد الزواج؟"
  },
  {
    "question_id": 888,
    "question": "ما حدود ما يجوز للخاطب النظر إليه من مخطوبته؟"
  },
  {
    "question_id": 889,
    "question": "هل يجوز للخاطب أن يختلي بمخطوبته قبل العقد؟"
  },
  {
    "question_id": 890,
    "question": "ما حكم زواج الرجل من امرأة طلقها ثلاثاً؟"
  },
  {
    "question_id": 891,
    "question": "هل يقع الطلاق إذا قال الزوج لزوجته أنت طالق مازحاً؟"
  },
  {
    "question_id": 892,
    "question": "ما حكم من طلق زوجته في حالة غضب شديد؟"
  },
  {
    "question_id": 893,
    "question": "هل يقع الطلاق إذا كتبه الزوج في رسالة هاتفية؟"
  },
  {
    "question_id": 894,
    "question": "ما حكم من حلف بالطلاق ثم حنث في يمينه؟"
  },
  {
    "question_id": 895,
    "question": "هل يقع الطلاق إذا قال الزوج لزوجته إن خرجت من البيت فأنت طالق؟"
  },
  {
    "question_id": 896,
    "question": "ما حكم الطلاق المعلق على شرط ثم لم يتحقق الشرط؟"
  },
  {
    "question_id": 897,
    "question": "هل يجوز للزوج أن يراجع زوجته بعد الطلقة الأولى دون عقد جديد؟"
  },
  {
    "question_id": 898,
    "question": "ما حكم رجوع الزوج إلى زوجته بعد انتهاء عدتها؟"
  },
  {
    "question_id": 899,
    "question": "كم عدة المرأة التي طلقها زوجها ولم يدخل بها؟"
  },
  {
    "question_id": 900,
    "question": "ما حكم المرأة التي طلقها زوجها وهي حامل؟"
  },
  {
    "question_id": 901,
    "question": "هل يجوز للزوجة طلب الطلاق بسبب سوء معاملة الزوج؟"
  },
  {
    "question_id": 902,
    "question": "ما حكم الخلع إذا طلبته المرأة دون وجود ضرر ظاهر؟"
  },
  {
    "question_id": 903,
    "question": "هل تجب النفقة على الزوج إذا كانت زوجته تعمل ولها راتب؟"
  },
  {
    "question_id": 904,
    "question": "ما حكم منع الزوج زوجته من زيارة والديها؟"
  },
  {
    "question_id": 905,
    "question": "هل يجوز للزوج أن يأخذ من راتب زوجته دون رضاها؟"
  },
  {
    "question_id": 906,
    "question": "ما حكم هجر الزوج زوجته مدة طويلة دون عذر؟"
  },
  {
    "question_id": 907,
    "question": "هل يجب على الأب العدل بين أولاده في الهدايا؟"
  },
  {
    "question_id": 908,
    "question": "هل يجب على الابن الإنفاق على والديه إذا كان فقيراً؟"
  },
  {
    "question_id": 909,
    "question": "ما حكم من حلف ألا يفعل شيئاً ثم فعل ذلك ناسياً يمينه؟"
  },
  {
    "question_id": 910,
    "question": "هل يجوز إخراج قيمة الطعام نقداً في كفارة اليمين؟"
  },
  {
    "question_id": 911,
    "question": "هل تنعقد اليمين إذا قالها الإنسان دون أن يقصد الحلف؟"
  },
  {
    "question_id": 912,
    "question": "ما حكم من حلف كذباً ليأخذ حقاً يظن أنه له؟"
  },
  {
    "question_id": 913,
    "question": "هل يجب الوفاء بالنذر إذا كان في معصية؟"
  },
  {
    "question_id": 914,
    "question": "ما حكم من نذر صيام شهر ثم لم يستطع الوفاء بنذره؟"
  },
  {
    "question_id": 915,
    "question": "هل يجوز تغيير النذر من صدقة إلى عمل خيري آخر؟"
  },
  {
    "question_id": 916,
    "question": "ما حكم من نذر ذبح شاة ولم يحدد لمن تكون؟"
  },
  {
    "question_id": 917,
    "question": "هل يجوز للزوج أن يغسل زوجته بعد وفاتها؟"
  },
  {
    "question_id": 918,
    "question": "هل يجوز للنساء اتباع الجنائز إلى المقبرة؟"
  },
  {
    "question_id": 919,
    "question": "هل يجوز تخصيص يوم معين من الأسبوع لزيارة المقابر؟"
  },
  {
    "question_id": 920,
    "question": "هل يجوز إهداء ثواب قراءة القرآن للميت؟"
  },
  {
    "question_id": 921,
    "question": "ما حكم الدعاء للميت بعد دفنه؟"
  },
  {
    "question_id": 922,
    "question": "ما حكم لبس الرجل للثوب الذي ينزل أسفل الكعبين؟"
  },
  {
    "question_id": 923,
    "question": "هل يجوز للرجل لبس خاتم الذهب؟"
  },
  {
    "question_id": 924,
    "question": "هل يجوز للمرأة لبس الملابس الضيقة أمام النساء؟"
  },
  {
    "question_id": 925,
    "question": "ما حكم صبغ الشعر باللون الأسود؟"
  },
  {
    "question_id": 926,
    "question": "هل يجوز للمرأة وصل شعرها بشعر صناعي؟"
  },
  {
    "question_id": 927,
    "question": "ما حكم الوشم المؤقت الذي يزول بعد مدة؟"
  },
  {
    "question_id": 928,
    "question": "هل يجوز للمرأة وضع العطر عند خروجها إلى الأماكن العامة؟"
  },
  {
    "question_id": 929,
    "question": "لو نسيت قراءة الفاتحة في ركعة، هل تبطل صلاتي؟"
  },
  {
    "question_id": 930,
    "question": "إذا تكلمت أثناء الصلاة ناسياً، فهل أعيد الصلاة؟"
  },
  {
    "question_id": 931,
    "question": "لو تحركت كثيراً في الصلاة بسبب طفل بجانبي، هل تبطل؟"
  },
  {
    "question_id": 932,
    "question": "إذا نسيت التشهد الأول ولم أسجد للسهو، فماذا أفعل؟"
  },
  {
    "question_id": 933,
    "question": "لو سلمت بعد الركعة الثالثة ظناً أنها الرابعة، ماذا أفعل؟"
  },
  {
    "question_id": 934,
    "question": "إذا شككت في عدد الركعات بعد السلام، هل أعيد الصلاة؟"
  },
  {
    "question_id": 935,
    "question": "لو دخلت المسجد والإمام راكع، هل أدركت الركعة؟"
  },
  {
    "question_id": 936,
    "question": "إذا نسيت صلاة الظهر وتذكرتها وقت العصر، بأي صلاة أبدأ؟"
  },
  {
    "question_id": 937,
    "question": "لو صليت ثم اكتشفت أن على ملابسي نجاسة، هل أعيد الصلاة؟"
  },
  {
    "question_id": 938,
    "question": "إذا صليت في اتجاه خاطئ ثم عرفت القبلة، هل أعيد الصلاة؟"
  },
  {
    "question_id": 939,
    "question": "لو غلبني النوم عن صلاة الفجر، هل علي إثم؟"
  },
  {
    "question_id": 940,
    "question": "إذا فاتتني صلاة الوتر ونمت عنها، متى أقضيها؟"
  },
  {
    "question_id": 941,
    "question": "لو خرج مني دم أثناء الصلاة، هل أقطع الصلاة؟"
  },
  {
    "question_id": 942,
    "question": "إذا انتقض وضوئي أثناء الصلاة، هل أستطيع إكمالها؟"
  },
  {
    "question_id": 943,
    "question": "لو مسحت على الجوربين ثم انتهت مدة المسح، هل أعيد الوضوء؟"
  },
  {
    "question_id": 944,
    "question": "إذا شككت هل اغتسلت من الجنابة أم لا، ماذا أفعل؟"
  },
  {
    "question_id": 945,
    "question": "لو توضأت ثم وضعت طلاء الأظافر، هل تصح صلاتي؟"
  },
  {
    "question_id": 946,
    "question": "إذا نزل مني المذي ولم أجد ماءً، هل يجوز لي التيمم؟"
  },
  {
    "question_id": 947,
    "question": "لو لم أستطع الوقوف بسبب ألم الظهر، كيف أصلي؟"
  },
  {
    "question_id": 948,
    "question": "إذا دخلت المسجد والإمام في التشهد الأخير، هل أصلي معه؟"
  },
  {
    "question_id": 949,
    "question": "لو نسيت سجود السهو وتذكرت بعد وقت طويل، ماذا أفعل؟"
  },
  {
    "question_id": 950,
    "question": "إذا قرأت الفاتحة بسرعة ولم أطمئن في الصلاة، هل تصح صلاتي؟"
  }
]

print(f"✓ Loaded {len(QUESTIONS_200):,} questions (IDs {QUESTIONS_200[0]['question_id']}..{QUESTIONS_200[-1]['question_id']}).")


In [ ]:
# 4. Load Qwen 0.6B Embedder and Mizan Reranker on GPU
from sentence_transformers import SentenceTransformer, CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"1. Loading Qwen 0.6B embedder on [{device}]...")
embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=device)
embedder.max_seq_length = 1024

print(f"2. Loading Mizan reranker on [{device}]...")
reranker = CrossEncoder("ALJIACHI/Mizan-Rerank-V2", max_length=1024, device=device, trust_remote_code=True)

print("3. Preparing GPU corpus index...")
corpus_embs_gpu = corpus_embeddings.to(device).float()
corpus_embs_gpu = torch.nn.functional.normalize(corpus_embs_gpu, p=2, dim=1)
print("✓ Models and index ready on GPU!")


In [ ]:
# 5. Generate 1,600 High-Yield Candidates (200 Questions x 8 Candidates from Big 3)
BIG3_QUOTAS = {
    "Ibn Baz": 4,
    "Ibn Uthaymeen": 2,
    "Al-Lajnah Ad-Daimah": 2,
}

queries_text = [q["question"] for q in QUESTIONS_200]
print(f"Encoding {len(queries_text)} queries on GPU with Qwen 0.6B...")
query_embs = embedder.encode(queries_text, batch_size=32, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
query_embs = query_embs.to(corpus_embs_gpu.dtype)

print("Computing similarity matrix...")
similarity_matrix = torch.matmul(query_embs, corpus_embs_gpu.T).cpu().numpy()

scholar_indices = {}
for scholar in BIG3_QUOTAS:
    scholar_indices[scholar] = np.where(corpus_df["source"].values == scholar)[0]

new_candidates_1600 = []
candidate_id_counter = 20001  # Safe ID prefix

print(f"\nRetrieving & Mizan-reranking 1,600 candidates across the Big 3 scholars...")
t0 = time.time()

for q_idx, q_item in enumerate(tqdm(QUESTIONS_200, desc="Generating Candidates")):
    q_id = q_item["question_id"]
    q_text = q_item["question"]
    q_sims = similarity_matrix[q_idx]
    
    for scholar, quota in BIG3_QUOTAS.items():
        idx_pool = scholar_indices.get(scholar)
        if idx_pool is None or len(idx_pool) == 0:
            continue
            
        pool_size = min(len(idx_pool), max(quota * 3, 10))
        top_sim_subindices = np.argpartition(-q_sims[idx_pool], pool_size - 1)[:pool_size]
        top_corpus_indices = idx_pool[top_sim_subindices]
        
        pairs = []
        for c_idx in top_corpus_indices:
            row = corpus_df.iloc[c_idx]
            ans_snippet = str(row.get("answer_text", ""))[:1200]
            pairs.append([q_text, f"{row.get('title', '')}\n{ans_snippet}"])
            
        scores = reranker.predict(pairs, show_progress_bar=False, batch_size=32)
        sorted_order = np.argsort(-scores)
        
        for rank, rank_idx in enumerate(sorted_order[:quota], start=1):
            c_idx = top_corpus_indices[rank_idx]
            row = corpus_df.iloc[c_idx]
            
            new_candidates_1600.append({
                "id": candidate_id_counter,
                "question_id": q_id,
                "category": "الفقه والعبادات",
                "question": q_text,
                "scholar": scholar,
                "fatwa_id": str(row.get("fatwa_id", "")),
                "title": str(row.get("title", "")),
                "question_text": str(row.get("question_text", "")),
                "answer_text": str(row.get("answer_text", "")),
                "reranker_score": float(scores[rank_idx]),
                "embedding_score": float(q_sims[c_idx]),
                "rank_in_scholar": rank,
                "dataset_split": "big3_expansion",
            })
            candidate_id_counter += 1

print(f"\n✓ Generated {len(new_candidates_1600):,} candidates in {time.time()-t0:.1f}s!")


In [ ]:
# 6. Ingest Previously Labeled Dataset (Loads existing ~2,049 positives & negatives)
existing_labeled_items = []

search_paths = [
    Path("/kaggle/input/dataset_9k_bundle.zip"),
    Path("/kaggle/input/dataset_9k_labeled.json"),
    Path("/kaggle/input/dataset_9k_labeled.csv"),
    Path("/kaggle/input/dataset_9k_full_raw.csv"),
    Path("/kaggle/working/dataset_9k_bundle.zip"),
    Path("/kaggle/working/dataset_9k_labeled.json"),
    Path("/kaggle/input/candidates_6k_bundle.zip"),
    Path("/kaggle/working/candidates_6k_bundle.zip"),
]
for p in Path("/kaggle/input").rglob("*"):
    if p.name in ("dataset_9k_bundle.zip", "dataset_9k_labeled.json", "dataset_9k_full_raw.csv", "candidates_6k_bundle.zip"):
        search_paths.insert(0, p)

found_file = None
for sp in search_paths:
    if sp.exists():
        found_file = sp
        break

assert found_file is not None, "Please attach dataset_9k_bundle.zip or candidates_6k_bundle.zip under Input!"
print(f"✓ Found previous dataset at: {found_file}")

if found_file.name.endswith(".zip"):
    with zipfile.ZipFile(found_file, "r") as zf:
        # Check files inside zip
        names = zf.namelist()
        target_name = None
        for n in ["dataset_9k_labeled.json", "dataset_9k_full_raw.csv", "dataset_9k_labeled.csv", "official_labeled_3k.json"]:
            if n in names:
                target_name = n
                break
        assert target_name, f"Could not find labeled data in zip: {names}"
        print(f"  • Reading {target_name} from zip...")
        with zf.open(target_name) as f:
            if target_name.endswith(".json"):
                existing_labeled_items = json.load(f)
            else:
                existing_labeled_items = pd.read_csv(f).to_dict(orient="records")
elif found_file.name.endswith(".json"):
    with open(found_file, "r", encoding="utf-8") as f:
        existing_labeled_items = json.load(f)
elif found_file.name.endswith(".csv"):
    existing_labeled_items = pd.read_csv(found_file).to_dict(orient="records")

if isinstance(existing_labeled_items, dict):
    existing_labeled_items = list(existing_labeled_items.values())

# Filter only items with valid labels
existing_labeled_items = [it for it in existing_labeled_items if str(it.get("gpt_verdict", "")).strip() in ("answers-it", "does-not-answer")]

old_pos = sum(1 for it in existing_labeled_items if it["gpt_verdict"] == "answers-it")
old_neg = len(existing_labeled_items) - old_pos
print(f"✓ Loaded {len(existing_labeled_items):,} previously labeled items:")
print(f"  • Positives: {old_pos:,} ({100*old_pos/len(existing_labeled_items):.1f}%)")
print(f"  • Negatives: {old_neg:,} ({100*old_neg/len(existing_labeled_items):.1f}%)")


In [ ]:
# 7. Configure OpenRouter API Key
import os
import getpass

OPENROUTER_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    OPENROUTER_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    if OPENROUTER_KEY:
        print("✓ Loaded OpenRouter API key from Kaggle Secrets.")
except Exception:
    pass

if not OPENROUTER_KEY:
    OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_KEY:
    OPENROUTER_KEY = getpass.getpass("🔑 Enter your OpenRouter API Key: ").strip()

assert OPENROUTER_KEY and OPENROUTER_KEY.startswith("sk-"), "Invalid OpenRouter API key!"
print("✓ OpenRouter API key verified.")


In [ ]:
# 8. Concurrent GPT-4o Mini Labeling of the 1,600 New Candidates (~$0.15, ~2.5 mins)
import requests
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / "new_1600_labeled_checkpoint.json"

labeled_new_cache = {}
if CHECKPOINT_FILE.exists():
    try:
        with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
            labeled_new_cache = json.load(f)
        print(f"✓ Found existing checkpoint: {len(labeled_new_cache):,} items already labeled!")
    except Exception:
        pass

def save_new_checkpoint(cache: dict):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

SYS_PROMPT = (
    "You are an expert Islamic fatwa relevance evaluation judge.\n"
    "Classify whether the provided Islamic fatwa directly and specifically answers the user's question.\n"
    "You must return ONLY ONE of the following two verdicts:\n"
    "1. 'answers-it': The fatwa directly and specifically answers the question asked.\n"
    "2. 'does-not-answer': The fatwa does NOT directly answer the question (including when it only touches on a related topic without directly answering, or is unrelated).\n\n"
    "Return a JSON object with:\n"
    "- 'verdict': exactly 'answers-it' or 'does-not-answer'\n"
    "- 'confidence': float between 0.0 and 1.0\n"
    "- 'explanation': brief explanation in Arabic."
)

def label_single_item(item: dict, max_retries: int = 3) -> dict:
    user_prompt = (
        f"User Question: {item['question']}\n\n"
        f"Scholar: {item['scholar']}\n"
        f"Title: {item['title']}\n"
        f"Fatwa Text: {item['answer_text'][:1400]}"
    )
    payload = {
        "model": "openai/gpt-4o-mini",
        "messages": [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": 0.0,
        "response_format": {"type": "json_object"},
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://kaggle.com",
        "X-Title": "Fatwa-RAG-Eval-Big3",
    }
    
    last_err = None
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=45)
            dt = time.perf_counter() - t0
            
            if resp.status_code == 429:
                time.sleep(2 * (attempt + 1))
                continue
            if resp.status_code >= 400:
                raise RuntimeError(f"API Error {resp.status_code}: {resp.text[:200]}")
            
            data = resp.json()
            content = data["choices"][0]["message"]["content"]
            usage = data.get("usage", {}) or {}
            
            verdict = "does-not-answer"
            conf = 1.0
            expl = ""
            try:
                m = re.search(r"\{.*?\}", content, re.DOTALL)
                if m:
                    parsed = json.loads(m.group(0))
                    raw_v = str(parsed.get("verdict", "")).lower().replace("_", "-").strip()
                    if raw_v in ("answers-it", "answers", "yes", "answers_it"):
                        verdict = "answers-it"
                    else:
                        verdict = "does-not-answer"
                    conf = float(parsed.get("confidence", 1.0))
                    expl = str(parsed.get("explanation", ""))
            except Exception:
                c_low = content.lower()
                if "answers-it" in c_low or "answers it" in c_low:
                    verdict = "answers-it"
                else:
                    verdict = "does-not-answer"
                
            return {
                "verdict": verdict,
                "confidence": conf,
                "explanation": expl,
                "cost": float(usage.get("cost", 0.0)),
                "latency_s": dt,
            }
        except Exception as e:
            last_err = e
            time.sleep(1.5 * (attempt + 1))
            
    raise RuntimeError(f"Failed after {max_retries} attempts: {last_err}")

# Filter pending items
pending_items = [it for it in new_candidates_1600 if str(it["id"]) not in labeled_new_cache]
print(f"Items to label: {len(pending_items):,} (Estimated Cost: ${len(pending_items) * 0.000095:.2f})")

MAX_WORKERS = 10
checkpoint_interval = 25
count_since_save = 0

if pending_items:
    pbar = tqdm(total=len(pending_items), desc="Labeling 1,600 Big 3 Candidates")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_item = {executor.submit(label_single_item, it): it for it in pending_items}
        for future in as_completed(future_to_item):
            it = future_to_item[future]
            cid = str(it["id"])
            try:
                res = future.result()
                labeled_new_cache[cid] = {
                    **it,
                    "gpt_verdict": res["verdict"],
                    "gpt_confidence": res["confidence"],
                    "gpt_explanation": res["explanation"],
                    "cost": res["cost"],
                    "latency_s": res["latency_s"],
                }
                count_since_save += 1
                if count_since_save >= checkpoint_interval:
                    save_new_checkpoint(labeled_new_cache)
                    count_since_save = 0
            except Exception as e:
                print(f"\nError: {e}")
            pbar.update(1)
            
    save_new_checkpoint(labeled_new_cache)
    print(f"\n✓ All {len(labeled_new_cache):,} new items successfully labeled!")


In [ ]:
# 9. Combine Pools & Build the Final 6,000+ Perfectly Balanced Dataset
# Step A: Update new items with their labels
new_labeled_list = list(labeled_new_cache.values())

new_pos = sum(1 for it in new_labeled_list if it["gpt_verdict"] == "answers-it")
new_neg = len(new_labeled_list) - new_pos
print("=" * 80)
print(f"NEW BIG 3 BATCH RESULTS ({len(new_labeled_list):,} items):")
print(f"  • Positives ('answers-it'):     {new_pos:,} ({100*new_pos/len(new_labeled_list):.1f}% hit rate!) 🎯")
print(f"  • Negatives ('does-not-answer'): {new_neg:,} ({100*new_neg/len(new_labeled_list):.1f}%)")
print("=" * 80)

# Step B: Merge existing labeled items + new Big 3 labeled items
total_combined_pool = existing_labeled_items + new_labeled_list
df_combined = pd.DataFrame(total_combined_pool)

df_all_positives = df_combined[df_combined["gpt_verdict"] == "answers-it"]
df_all_negatives = df_combined[df_combined["gpt_verdict"] == "does-not-answer"]

total_positives_count = len(df_all_positives)
print(f"\nTOTAL POSITIVES IN HAND: {total_positives_count:,} (Old: {old_pos:,} + New: {new_pos:,})")
print(f"TOTAL NEGATIVES IN HAND: {len(df_all_negatives):,}")

# Step C: Select the best matching Hard Negatives (stratified by scholar, highest Mizan score)
# Target ratio: 48% - 50% positive (take ~1.05x to 1.1x negatives)
target_negatives_count = int(total_positives_count * 1.08)

# Group by scholar and pick highest reranker scores
df_hard_negatives = df_all_negatives.sort_values(by="reranker_score", ascending=False).groupby(
    "scholar", group_keys=False
).apply(lambda x: x.head(int(target_negatives_count / df_combined["scholar"].nunique()) + 100)).head(target_negatives_count)

# Step D: Assemble the balanced dataset
df_final_balanced = pd.concat([df_all_positives, df_hard_negatives]).sample(frac=1.0, random_state=42).reset_index(drop=True)
for i in range(len(df_final_balanced)):
    df_final_balanced.at[i, "id"] = i + 1

final_pos = (df_final_balanced["gpt_verdict"] == "answers-it").sum()
final_neg = (df_final_balanced["gpt_verdict"] == "does-not-answer").sum()

print("\n" + "=" * 80)
print("🎉 FINAL BALANCED DATASET CREATED!")
print("=" * 80)
print(f"Total Balanced Examples:        {len(df_final_balanced):,} items")
print(f"Positives ('answers-it'):        {final_pos:,} ({100*final_pos/len(df_final_balanced):.1f}%)")
print(f"Negatives ('does-not-answer'):    {final_neg:,} ({100*final_neg/len(df_final_balanced):.1f}%)")
print("=" * 80)

print("\nDISTRIBUTION BY SCHOLAR IN BALANCED DATASET:")
scholar_crosstab = pd.crosstab(df_final_balanced["scholar"], df_final_balanced["gpt_verdict"], margins=True)
display(scholar_crosstab)

# Step E: Export files and bundle zip
df_final_balanced.to_csv(OUTPUT_DIR / "dataset_6k_balanced.csv", index=False, encoding="utf-8")
df_final_balanced.to_json(OUTPUT_DIR / "dataset_6k_balanced.json", orient="records", force_ascii=False, indent=2)

zip_path = OUTPUT_DIR / "dataset_6k_balanced_bundle.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUTPUT_DIR / "dataset_6k_balanced.csv", arcname="dataset_6k_balanced.csv")
    zf.write(OUTPUT_DIR / "dataset_6k_balanced.json", arcname="dataset_6k_balanced.json")

print("\n" + "=" * 80)
print(f"✓ Archive created: {zip_path.name} ({zip_path.stat().st_size / (1024**2):.2f} MB)")
print("Download dataset_6k_balanced_bundle.zip to proceed with training Notebook 2!")
print("=" * 80)
